In [1]:
import pandas as pd 
df = pd.read_csv('Covid19_bal.csv') 

from sklearn.model_selection import train_test_split 
X_train_text, X_test_text, y_train, y_test = train_test_split( df['title'], df['label'], test_size=0.2, random_state=42 )

In [2]:
# =========================
# Fine-tune BERT
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))  # or len(set(df['label']))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)
model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.1817


Training Epoch 2:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.0459


...

Epoch 3 | Train Loss: 0.0130


Training Epoch 4:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.0046


Evaluating:   0%|          | 0/123 [00:00<?, ?it/s]


Validation Accuracy: 0.9791 | F1-score: 0.9791

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       983
           1       0.98      0.98      0.98       979

    accuracy                           0.98      1962
   macro avg       0.98      0.98      0.98      1962
weighted avg       0.98      0.98      0.98      1962


Confusion Matrix:
[[966  17]
 [ 24 955]]


In [3]:
# =========================
# Fine-tune RoBERTa
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss / len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.1731


Training Epoch 2:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.0573


...

Epoch 3 | Train Loss: 0.0260


Training Epoch 4:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.0095


Evaluating:   0%|          | 0/123 [00:00<?, ?it/s]


Validation Accuracy: 0.9771 | F1-score: 0.9771

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       983
           1       0.99      0.96      0.98       979

    accuracy                           0.98      1962
   macro avg       0.98      0.98      0.98      1962
weighted avg       0.98      0.98      0.98      1962


Confusion Matrix:
[[973  10]
 [ 35 944]]


In [4]:
import pickle

with open("best_transformer_model_roberta.pkl", "wb") as f:
    pickle.dump(model, f)

with open("roberta_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [5]:
save_path = "roberta_finetuned_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")

Model saved successfully!


In [6]:
# =========================
# Fine-tune ALBERT
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AlbertTokenizer,
    AlbertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')

model = AlbertForSequenceClassification.from_pretrained(
    'albert-base-v2',
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss / len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.1990


Training Epoch 2:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.0708


...

Epoch 3 | Train Loss: 0.0293


Training Epoch 4:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.0105


Evaluating:   0%|          | 0/123 [00:00<?, ?it/s]


Validation Accuracy: 0.9760 | F1-score: 0.9760

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       983
           1       0.98      0.97      0.98       979

    accuracy                           0.98      1962
   macro avg       0.98      0.98      0.98      1962
weighted avg       0.98      0.98      0.98      1962


Confusion Matrix:
[[966  17]
 [ 30 949]]


In [7]:
# =========================
# Fine-tune RoBERTa
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 5
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss / len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.1702


Training Epoch 2:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.0509


...

Epoch 4 | Train Loss: 0.0093


Training Epoch 5:   0%|          | 0/491 [00:00<?, ?it/s]

Epoch 5 | Train Loss: 0.0032


Evaluating:   0%|          | 0/123 [00:00<?, ?it/s]


Validation Accuracy: 0.9827 | F1-score: 0.9827

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       983
           1       0.99      0.97      0.98       979

    accuracy                           0.98      1962
   macro avg       0.98      0.98      0.98      1962
weighted avg       0.98      0.98      0.98      1962


Confusion Matrix:
[[976   7]
 [ 27 952]]


In [9]:
import pickle

with open("best_transformer_model_roberta.pkl", "wb") as f:
    pickle.dump(model, f)

with open("roberta_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [10]:
save_path = "roberta_finetuned_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")

Model saved successfully!
